In [1]:
from minsearch import vector
from prompt_toolkit import document
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
v1.dot(dv)

np.float32(0.323324)

In [5]:
from Module01_AgenticRAG.ingest import load_faq_data

In [6]:
documents = load_faq_data()

In [7]:
documents[2]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [8]:
texts = []
for doc in documents:
    text = doc['question']+' '+doc['answer']
    texts.append(text)


In [9]:
texts[2]

"Course: Can I still join the course after the start date? Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."

In [10]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_vector = model.encode(batch)
    vectors.extend(batch_vector)
len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1368

In [11]:
import numpy as np
X = np.array(vectors)

In [15]:
scores = X.dot(v1)
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629412))

In [16]:
texts[idx]

"Course: Can I still join the course after the start date? Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."

In [17]:
scores = X.dot(dv)
idx = np.argmax(scores)
idx, scores[idx]



(np.int64(906), np.float32(0.6439043))

In [18]:
texts[idx]

'Do I need to enroll in the course before submitting homework? No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'

In [33]:
scores = X.dot(v1)
idx = np.argsort(scores)
idx = idx[-5:]
idx, scores[idx]

(array([  7, 538, 925, 643,   2]),
 array([0.56009996, 0.65363115, 0.7192131 , 0.7579371 , 0.7629412 ],
       dtype=float32))

In [34]:
len(idx)

5

In [36]:
for id in idx:
    print(scores[id])
    print(texts[id])

0.56009996
Course - Can I follow the course after it finishes? Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.

You can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.
0.65363115
I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
0.7192131
The course has already started. Can I still join it? Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.

In order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.
0.757937

In [38]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X,documents)

In [42]:
vindex.search(v1,num_results=5, filter_dict={'course':'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for